***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [9. 实践部分](9_1_visualisation-inspection.ipynb)
    * 上一节： [9.14 端到端连续谱教学案例](9_14_end_to_end_continuum_case.ipynb)
    * 下一节： [9.x 延伸阅读与后续实践方向](9_x_further_reading_and_workflow.ipynb)

***


## 9.15 连续谱源表生成：PyBDSF 风格的图像到目录流程

连续谱成像的终点常常不是一张图，而是一张可用于统计、交叉证认和物理解释的源表。巡天项目需要源表来做源计数、光度函数、AGN 与恒星形成星系分类；多波段研究需要源表来和红外、光学、X 射线目录交叉匹配；单个目标场研究也常常需要把图像中的紧致源、展源和残余伪影分开。源表因此不是图像之后的附属表格，而是从图像走向科学量的关键数据产品。

PyBDSF 正是这一环节中常用的连续谱源搜索工具。它的核心任务，是从恢复图像及相关诊断产品中估计背景和局部 RMS，识别显著的 island，把 island 分解为高斯 components，再把 components 组织为 sources 并输出目录、模型图和残差图。本节不依赖本地安装 PyBDSF，也不把任何参数当作固定配方；这里采用 PyBDSF 风格的简化案例，目的是把源表生成背后的统计与图像判断讲清楚。


### 9.15.1 从图像到源表，首先是背景和噪声问题

源检测的第一步不是寻找峰值，而是建立图像中“什么叫显著”的局部标准。干涉图像的噪声通常不是处处相同的。主波束校正会放大视场边缘噪声，亮源附近的去卷积残差会提高局部 RMS，mosaic 权重和方向相关误差也会造成位置依赖灵敏度。若只用一个全局 RMS 来定义阈值，安静区域中的弱源可能被漏掉，嘈杂区域中的噪声峰又可能被误检为源。

因此，PyBDSF 风格的源表流程通常先估计背景图和 RMS 图，再在局部信噪比图上进行检测。背景图描述缓慢变化的零点，RMS 图描述局部噪声尺度，二者共同定义每个像素的显著性。这个步骤的质量决定了后面所有阈值和源表参数的可信度。


![连续谱图像、局部 RMS 与信噪比图](figures/practical_source_catalog_image_noise_snr.png)

**图 9.15.1** 一个含紧致源、双源、展源和位置相关噪声的连续谱图像。左图为恢复图像，中图为局部 RMS 图，右图为局部 S/N 图。视场边缘和亮源附近的噪声不同，说明源检测应在局部噪声标尺下进行。


### 9.15.2 Island、component 与 source 不是同一个层级

连续谱源表中经常出现三个容易混淆的层级。第一层是 island，即一片相连的显著像素区域。它由像素阈值和 island 阈值共同定义：较高的像素阈值确认区域内至少有一个足够显著的峰，较低的 island 阈值决定这个候选区域的边界。第二层是 component，通常用一个或多个高斯分量描述 island 内的亮度结构。第三层是 source，即科学目录中希望表示的天体对象；一个 source 可能由一个 component 构成，也可能由多个 components 构成，例如双瓣射电星系或核心-喷流结构。

这个分层很重要，因为源表中的“源数”并不等于高斯分量数。高分辨率图像会把一个真实天体拆成多个亮斑，低分辨率图像又可能把相邻天体合并成一个 island。目录的科学解释必须说明使用的是 component catalogue 还是 source catalogue，以及合并规则是什么。


![连续谱源搜索中的 island、component 和 residual](figures/practical_source_catalog_islands_components_residual.png)

**图 9.15.2** 简化 PyBDSF 风格源搜索。左图显示检测到的 islands 和拟合得到的高斯 components，中图为 component model，右图为 residual image。残差图是判断分解是否合理的核心诊断，不应只看输出目录。


### 9.15.3 阈值选择是一种科学取舍

源搜索参数没有脱离科学目标的“最佳值”。较松的阈值可以提高 completeness，即恢复更多真实弱源，但也会增加噪声峰和残余旁瓣进入目录的概率，降低 reliability。较严格的阈值可以提高目录可靠性，却会牺牲弱源、低面亮度源和视场边缘源。对于巡天统计，completeness 和 reliability 的平衡必须通过注入源恢复、负源统计或人工验证样本来量化；对于单个目标场，阈值选择还应结合源形态和科学目标。

PyBDSF 中常见的 `thresh_pix` 和 `thresh_isl` 正好体现了这种双阈值思想。`thresh_pix` 决定 island 中是否存在显著峰，`thresh_isl` 决定 island 的边界生长到哪里。两个参数共同影响源的检测数、积分通量、分量拆分和残差结构。把它们当作固定默认值使用，很容易在不同噪声环境或不同分辨率图像中产生系统偏差。


![源搜索阈值对 completeness 和 reliability 的影响](figures/practical_source_catalog_threshold_tradeoff.png)

**图 9.15.3** 阈值选择的典型取舍。较松阈值可以恢复更多候选源，但伪检数增加；较严格阈值更可靠，却会漏掉弱源。源表参数应服务科学目标，而不是机械追随默认值。


### 9.15.4 源表质量控制：目录、模型图和残差图必须一起看

一个源表是否可信，不能只由检测源数或表格格式判断。至少需要同时检查四类诊断。第一，残差图是否仍有系统性结构；若亮源附近残留明显正负条纹，源表中靠近这些结构的候选源可靠性很低。第二，负源统计是否合理；对于 Stokes I 连续谱图像，真实天体主要贡献正流量，因此显著负 island 的数量可以作为伪检率的一个诊断。第三，局部 RMS 图是否被真实源或残余伪影污染；若噪声图在源附近被抬高，弱源显著性会被低估。第四，component model 是否过拟合；过多高斯分量可能把噪声和去卷积残差吸收入模型。

这些检查与第 9.14 节的端到端案例直接相连。校准残差、CLEAN mask、主波束模型和自校准策略都会改变源搜索的输入图像。若前面的图像产品没有达到可解释状态，后面的源表也不会因为使用了成熟工具而自动可信。


![连续谱源表的残差、负源和目录质量检查](figures/practical_source_catalog_validation_checks.png)

**图 9.15.4** 源表质量控制示意。残差 S/N 直方图、负源检查、源表摘要和人工检查样本共同用于判断目录可靠性。源表验证应关注失败模式，而不只是确认成功检测。


### 9.15.5 与真实 PyBDSF 工作流的对应

真实使用 PyBDSF 时，常见入口是对恢复后的 FITS 图像运行 `process_image`，并设置与检测和背景估计相关的参数，例如背景/RMS box、`thresh_pix`、`thresh_isl`、是否输出 island mask、model image、residual image 和 catalogue。PyBDSF 还支持更复杂的源结构处理，例如多高斯分解、形态复杂源的进一步建模，以及在多频或偏振场景下输出更丰富的源性质。官方文档入口是 <https://pybdsf.readthedocs.io/>。

教材中引入 PyBDSF 时，应避免把它写成“运行一次得到目录”的黑箱。更合适的教学方式，是先用本节这种可解释原型建立概念，再把概念映射到真实参数：背景/RMS 估计对应图 9.15.1，island 与 component 对应图 9.15.2，阈值选择对应图 9.15.3，残差和负源统计对应图 9.15.4。这样即使换用其他源搜索器，判断链仍然成立。


### 9.15.6 本节的教学重点

连续谱源表生成可以分成三个训练层次。基础层次应掌握图像、局部 RMS、S/N、island、component 和 source 的区别。中级层次应能解释阈值、局部噪声估计和高斯分解如何影响 completeness、reliability 和积分通量。研究训练层次还应进一步考虑注入源恢复、负源统计、主波束边缘、残余旁瓣、多分量射电星系和跨波段匹配中的系统误差。

本节最重要的结论是：源表不是图像中亮点的坐标列表，而是一个经过模型假设、阈值选择和质量验证的数据产品。PyBDSF 的价值，正在于它把这些步骤组织成可复现流程；教材的任务，则是把流程背后的物理和统计判断讲清楚。

***
